# Tick数据读取演示

本notebook示范如何读取tick_2026目录下的tick数据

In [2]:
# 导入模块
import os
os.chdir('/data1/code_git/tick_data_analysis')
from mylib.get_tick_data import get_tick_data, get_available_dates, get_available_stocks,get_tick_data_short
from mylib.get_local_data import get_local_data
from datetime import date
import pandas as pd

## 1. 查看可用数据

In [3]:
# 获取可用日期
dates = get_available_dates("2026")
print(f"可用日期数量: {len(dates)}")
print(f"前5个日期: {dates[:5]}")
print(f"最后5个日期: {dates[-5:]}")

可用日期数量: 46
前5个日期: [datetime.date(2026, 1, 5), datetime.date(2026, 1, 6), datetime.date(2026, 1, 7), datetime.date(2026, 1, 8), datetime.date(2026, 1, 9)]
最后5个日期: [datetime.date(2026, 3, 9), datetime.date(2026, 3, 10), datetime.date(2026, 3, 11), datetime.date(2026, 3, 12), datetime.date(2026, 3, 13)]


In [4]:
print('查看可转债前三个tick的值')
kzz_tick_df = pd.read_parquet('kzz_first_3_ticks_after_930_2026.parquet')
print(kzz_tick_df)

查看可转债前三个tick的值
             date stock_code  tick_seq          time    amount  diff_amount  \
0      2026-01-05  110067.SH         1  09:30:01.000   58712.0      36226.0   
1      2026-01-05  110067.SH         2  09:30:04.000   68703.0      46217.0   
2      2026-01-05  110067.SH         3  09:30:07.000   74933.0      52447.0   
3      2026-01-05  110070.SH         1  09:30:01.000  128856.0      43365.0   
4      2026-01-05  110070.SH         2  09:30:04.000  167228.0      81737.0   
...           ...        ...       ...           ...       ...          ...   
49837  2026-03-13  128141.SZ         2  09:30:03.000  398151.0     392696.0   
49838  2026-03-13  128141.SZ         3  09:30:06.000  428217.0     422762.0   
49839  2026-03-13  128142.SZ         1  09:30:00.000   29203.0      18572.0   
49840  2026-03-13  128142.SZ         2  09:30:03.000   56981.0      46350.0   
49841  2026-03-13  128142.SZ         3  09:30:06.000   86115.0      75484.0   

       lastPrice  lastClose  pct_cha

In [ ]:
tick_a = {}
for n in [1,2,3]:
    tick_a[n] = kzz_tick_df[kzz_tick_df['tick_seq'] == n]
print(tick_a[1])

In [10]:
# 获取指定日期的股票列表
target_date = dates[0]
print(f"查看 {target_date} 的股票数据")
stocks = get_available_stocks(target_date)
print(f"股票数量: {len(stocks)}")
print(f"前10只股票: {stocks[:10]}")

查看 2026-01-05 的股票数据
股票数量: 5854
前10只股票: ['000001.SZ', '000002.SZ', '000004.SZ', '000006.SZ', '000007.SZ', '000008.SZ', '000009.SZ', '000010.SZ', '000011.SZ', '000012.SZ']


In [11]:
print('获取zz数据')
import tushare as ts
pro = ts.pro_api('6335cdd3c50d6fb7f682cb55b911561593a7434d64ec7bc937dbd9c0')
kzz_info = pro.cb_basic().set_index('ts_code')
kzz_list = kzz_info.index.tolist()
print(kzz_info)

获取zz数据
                             bond_full_name bond_short_name cb_code  \
ts_code                                                               
125002.SZ                 万科企业股份有限公司可转换公司债券            万科转债    None   
125009.SZ               中国宝安集团股份有限公司可转换公司债券            宝安转券    None   
125069.SZ              深圳华侨城控股股份有限公司可转换公司债券            侨城转债    None   
125301.SZ                 吴江丝绸股份有限公司可转换公司债券            丝绸转债    None   
126301.SZ                 吴江丝绸股份有限公司可转换公司债券            丝绸转2    None   
...                                     ...             ...     ...   
118065.SH     上海艾为电子技术股份有限公司向不特定对象发行可转换公司债券            艾为转债    None   
113700.SH         海天水务集团股份公司向不特定对象发行可转换公司债券            海天转债    None   
118066.SH  深圳市泛海统联精密制造股份有限公司向不特定对象发行可转换公司债券            统联转债    None   
113701.SH         浙江天台祥和实业股份有限公司公开发行可转换公司债券            祥和转债    None   
127113.SZ         长高电新科技股份公司向不特定对象发行可转换公司债券            长高转债    None   

            stk_code stk_short_name  maturity    par  issue_price  \


In [12]:
# kzz_daily_data = get_local_data(sec_list=kzz_list,start='20260101',end='20260313')
kzz_daily_data = get_local_data(
      kzz_list,
      '20260101',
      '20260312',
      'all',
      'kzz_daily',
  )
# temp_pqt_df = pd.read_parquet('daily_data/daily/2026_full.parquet')
# print(temp_pqt_df[temp_pqt_df['trade_date'] == '20260210'])
print(kzz_daily_data.keys())

[get_all_data] kzz_daily: 1年数据, 0个full文件, 1个需合并
dict_keys(['open', 'high', 'low', 'close', 'pre_close', 'change', 'pct_chg', 'vol', 'amount'])


In [13]:
print('最高价zz')
full_df = pd.DataFrame(kzz_daily_data['close'].stack())
full_df.columns = ['close']
full_df.index.names = ['day','sec']
top_full_df = full_df[full_df.groupby('day')['close'].rank(ascending=False)<=10]
temp_top_list = top_full_df.loc['2026-1-5'].index.tolist()
print(temp_top_list)
print(top_full_df.loc['2026-1-5'])

最高价zz
['110074.SH', '111012.SH', '113589.SH', '113615.SH', '113687.SH', '118058.SH', '123118.SZ', '123241.SZ', '127037.SZ', '127070.SZ']
              close
sec                
110074.SH   395.490
111012.SH   413.500
113589.SH   317.350
113615.SH   765.897
113687.SH   472.674
118058.SH   329.753
123118.SZ  1255.700
123241.SZ   622.200
127037.SZ   466.200
127070.SZ   330.500


In [ ]:
from mylib.get_tick_data import get_tick_data_short
top_tick_data = get_tick_data_short(temp_top_list,start_date='2026-01-06',end_date='2026-02-06')
# top_tick_data = get_tick_data(temp_top_list,start_date='2026-01-06',end_date='2026-01-06')
# top_tick_data = get_tick_data(['123118.SZ'],start_date='2026-01-06',end_date='2026-01-16')
# top_tick_data = get_tick_data(['118058.SH'],start_date='2026-01-06',end_date='2026-03-16')


dict_keys(['110074.SH', '111012.SH', '113589.SH', '113615.SH', '113687.SH', '118058.SH', '123118.SZ', '123241.SZ', '127037.SZ', '127070.SZ'])
110074.SH                               amount  diff_amount  lastPrice
datetime                                                    
2026-01-06 09:30:01+08:00  1580548.0     963538.0    395.974
2026-01-06 09:30:04+08:00  2952223.0    1371675.0    396.970
2026-01-06 09:30:07+08:00  6160983.0    3208760.0    395.247
2026-01-06 09:30:10+08:00  6931052.0     770069.0    394.518
2026-01-06 09:30:13+08:00  8673406.0    1742354.0    393.821
...                              ...          ...        ...
2026-02-06 09:31:47+08:00  6932785.0     147639.0    509.182
2026-02-06 09:31:50+08:00  6973535.0      40750.0    509.448
2026-02-06 09:31:53+08:00  7024486.0      50951.0    509.658
2026-02-06 09:31:56+08:00  7121319.0      96833.0    509.716
2026-02-06 09:31:59+08:00  7141697.0      20378.0    509.456

[968 rows x 3 columns]


In [27]:
print(top_tick_data.keys())
for k,v in top_tick_data.items():
    v['ret'] = v['lastPrice'] / v['lastClose']
    print(k,v.between_time('9:30','9:31').groupby('day')[['amount','diff_amount','lastPrice','ret']].head(5).head(30))
    # print(k,v.resample('30s').mean())
    break

dict_keys(['110074.SH', '111012.SH', '113589.SH', '113615.SH', '113687.SH', '118058.SH', '123118.SZ', '123241.SZ', '127037.SZ', '127070.SZ'])
110074.SH                                amount  diff_amount  lastPrice       ret
datetime                                                               
2026-01-06 09:30:01+08:00   1580548.0     963538.0    395.974  1.001224
2026-01-06 09:30:04+08:00   2952223.0    1371675.0    396.970  1.003742
2026-01-06 09:30:07+08:00   6160983.0    3208760.0    395.247  0.999386
2026-01-06 09:30:10+08:00   6931052.0     770069.0    394.518  0.997542
2026-01-06 09:30:13+08:00   8673406.0    1742354.0    393.821  0.995780
2026-01-07 09:30:01+08:00    815440.0     484159.0    406.746  0.998000
2026-01-07 09:30:04+08:00   1532158.0     716718.0    406.742  0.997990
2026-01-07 09:30:07+08:00   2580566.0    1048408.0    404.909  0.993493
2026-01-07 09:30:10+08:00   2924937.0     344371.0    404.994  0.993702
2026-01-07 09:30:13+08:00   3426301.0     501364.0    40

## 2. 读取单个股票数据

In [8]:
# 读取单个股票全部数据
df = get_tick_data("000001.SZ")
print(f"000001.SZ 总记录数: {len(df)}")
print(f"列名: {df.columns.tolist()}")
print(df.between_time('9:24','9:26'))

000001.SZ 总记录数: 693545
列名: ['time', 'lastPrice', 'open', 'high', 'low', 'lastClose', 'amount', 'volume', 'pvolume', 'tickvol', 'stockStatus', 'openInt', 'lastSettlementPrice', 'askPrice', 'bidPrice', 'askVol', 'bidVol', 'settlementPrice', 'transactionNum', 'pe', 'day', 'time_ms']
                                   time  lastPrice   open   high    low  \
datetime                                                                  
2025-07-31 09:24:03+08:00  09:24:03.000       0.00   0.00   0.00   0.00   
2025-07-31 09:24:12+08:00  09:24:12.000       0.00   0.00   0.00   0.00   
2025-07-31 09:24:21+08:00  09:24:21.000       0.00   0.00   0.00   0.00   
2025-07-31 09:24:30+08:00  09:24:30.000       0.00   0.00   0.00   0.00   
2025-07-31 09:24:39+08:00  09:24:39.000       0.00   0.00   0.00   0.00   
...                                 ...        ...    ...    ...    ...   
2026-03-10 09:24:30+08:00  09:24:30.000       0.00   0.00   0.00   0.00   
2026-03-10 09:24:39+08:00  09:24:39.000     

In [5]:
print(df)

                                   time  lastPrice   open   high    low  \
datetime                                                                  
2025-07-31 09:15:00+08:00  09:15:00.000       0.00   0.00   0.00   0.00   
2025-07-31 09:15:09+08:00  09:15:09.000       0.00   0.00   0.00   0.00   
2025-07-31 09:15:18+08:00  09:15:18.000       0.00   0.00   0.00   0.00   
2025-07-31 09:15:36+08:00  09:15:36.000       0.00   0.00   0.00   0.00   
2025-07-31 09:15:45+08:00  09:15:45.000       0.00   0.00   0.00   0.00   
...                                 ...        ...    ...    ...    ...   
2026-03-10 14:59:24+08:00  14:59:24.000      10.80  10.77  10.81  10.73   
2026-03-10 14:59:33+08:00  14:59:33.000      10.80  10.77  10.81  10.73   
2026-03-10 14:59:42+08:00  14:59:42.000      10.80  10.77  10.81  10.73   
2026-03-10 14:59:51+08:00  14:59:51.000      10.80  10.77  10.81  10.73   
2026-03-10 15:00:00+08:00  15:00:00.000      10.81  10.77  10.81  10.73   

                        

In [12]:
# 读取指定日期的数据
import numpy as np
df = get_tick_data("000001.SZ", start_date="20260105")
print(f"20260105 的记录数: {len(df)}")
# print(df)
df[['askPrice1','askPrice2','askPrice3','askPrice4','askPrice5']] = np.vstack(df['askPrice'].values)
print(df)
# print(df[[ 'lastPrice', 'volume', 'amount', 'bidPrice1', 'askPrice1']].head())

20260105 的记录数: 4811
                                   time  lastPrice   open   high    low  \
datetime                                                                  
2026-01-05 09:15:00+08:00  09:15:00.000       0.00   0.00   0.00   0.00   
2026-01-05 09:15:09+08:00  09:15:09.000       0.00   0.00   0.00   0.00   
2026-01-05 09:15:18+08:00  09:15:18.000       0.00   0.00   0.00   0.00   
2026-01-05 09:15:27+08:00  09:15:27.000       0.00   0.00   0.00   0.00   
2026-01-05 09:15:45+08:00  09:15:45.000       0.00   0.00   0.00   0.00   
...                                 ...        ...    ...    ...    ...   
2026-01-05 14:59:24+08:00  14:59:24.000      11.51  11.42  11.51  11.41   
2026-01-05 14:59:33+08:00  14:59:33.000      11.51  11.42  11.51  11.41   
2026-01-05 14:59:42+08:00  14:59:42.000      11.51  11.42  11.51  11.41   
2026-01-05 14:59:51+08:00  14:59:51.000      11.51  11.42  11.51  11.41   
2026-01-05 15:00:00+08:00  15:00:00.000      11.50  11.42  11.51  11.41   

    

In [14]:
print(df.between_time('9:30','9:31').iloc[-3:,:10])
print(df.between_time('9:30','9:31').iloc[-3:,10:])

                                   time  lastPrice   open   high    low  \
datetime                                                                  
2026-01-05 09:30:54+08:00  09:30:54.000      11.43  11.42  11.45  11.42   
2026-01-05 09:30:57+08:00  09:30:57.000      11.42  11.42  11.45  11.42   
2026-01-05 09:31:00+08:00  09:31:00.000      11.43  11.42  11.45  11.42   

                           lastClose      amount  volume  pvolume  tickvol  
datetime                                                                    
2026-01-05 09:30:54+08:00      11.41  21395886.0   18717        0        0  
2026-01-05 09:30:57+08:00      11.41  21630063.0   18922        0        0  
2026-01-05 09:31:00+08:00      11.41  21914861.0   19171        0        0  
                           stockStatus  openInt  lastSettlementPrice  \
datetime                                                               
2026-01-05 09:30:54+08:00            0       13                  0.0   
2026-01-05 09:30:57+08:

In [6]:
df = get_tick_data("000001.SZ", start_date="2026-01-05")
# print(df.between_time('9:20','9:26'))
print(df.iloc[20:40])

                                   time  lastPrice  open  high  low  \
datetime                                                              
2026-01-05 09:18:54+08:00  09:18:54.000        0.0   0.0   0.0  0.0   
2026-01-05 09:19:03+08:00  09:19:03.000        0.0   0.0   0.0  0.0   
2026-01-05 09:19:12+08:00  09:19:12.000        0.0   0.0   0.0  0.0   
2026-01-05 09:19:21+08:00  09:19:21.000        0.0   0.0   0.0  0.0   
2026-01-05 09:19:30+08:00  09:19:30.000        0.0   0.0   0.0  0.0   
2026-01-05 09:19:39+08:00  09:19:39.000        0.0   0.0   0.0  0.0   
2026-01-05 09:19:48+08:00  09:19:48.000        0.0   0.0   0.0  0.0   
2026-01-05 09:19:57+08:00  09:19:57.000        0.0   0.0   0.0  0.0   
2026-01-05 09:20:00+08:00  09:20:00.000        0.0   0.0   0.0  0.0   
2026-01-05 09:20:09+08:00  09:20:09.000        0.0   0.0   0.0  0.0   
2026-01-05 09:20:18+08:00  09:20:18.000        0.0   0.0   0.0  0.0   
2026-01-05 09:20:27+08:00  09:20:27.000        0.0   0.0   0.0  0.0   
2026-0

In [3]:
df = get_tick_data_short("000001.SZ", start_date="2026-01-05")
print(df.between_time('9:20','9:26'))

                                  day          time  lastPrice   open   high  \
datetime                                                                       
2026-01-05 09:20:00+08:00  2026-01-05  09:20:00.000       0.00   0.00   0.00   
2026-01-05 09:20:09+08:00  2026-01-05  09:20:09.000       0.00   0.00   0.00   
2026-01-05 09:20:18+08:00  2026-01-05  09:20:18.000       0.00   0.00   0.00   
2026-01-05 09:20:27+08:00  2026-01-05  09:20:27.000       0.00   0.00   0.00   
2026-01-05 09:20:36+08:00  2026-01-05  09:20:36.000       0.00   0.00   0.00   
2026-01-05 09:20:45+08:00  2026-01-05  09:20:45.000       0.00   0.00   0.00   
2026-01-05 09:20:54+08:00  2026-01-05  09:20:54.000       0.00   0.00   0.00   
2026-01-05 09:21:03+08:00  2026-01-05  09:21:03.000       0.00   0.00   0.00   
2026-01-05 09:21:12+08:00  2026-01-05  09:21:12.000       0.00   0.00   0.00   
2026-01-05 09:21:21+08:00  2026-01-05  09:21:21.000       0.00   0.00   0.00   
2026-01-05 09:21:30+08:00  2026-01-05  0

## 3. 读取日期范围

In [7]:
# 读取日期范围
df = get_tick_data("600235.SH", start_date="20260101", end_date="20260110")
# df.index = pd.to_datetime(df['time'],unit='ms',utc=True).dt.tz_convert('Asia/Shanghai')
# print(f"20260101-20260110 的记录数: {len(df)}")
# print(f"涵盖日期: {sorted(df['file_date'].unique())}")
print(df)

                                   time  lastPrice  open  high   low  \
datetime                                                               
2026-01-05 09:15:01+08:00  09:15:01.000       0.00  0.00  0.00  0.00   
2026-01-05 09:15:04+08:00  09:15:04.000       0.00  0.00  0.00  0.00   
2026-01-05 09:15:07+08:00  09:15:07.000       0.00  0.00  0.00  0.00   
2026-01-05 09:15:10+08:00  09:15:10.000       0.00  0.00  0.00  0.00   
2026-01-05 09:15:13+08:00  09:15:13.000       0.00  0.00  0.00  0.00   
...                                 ...        ...   ...   ...   ...   
2026-01-09 14:59:43+08:00  14:59:43.000       6.83  6.84  6.85  6.71   
2026-01-09 14:59:52+08:00  14:59:52.000       6.83  6.84  6.85  6.71   
2026-01-09 14:59:55+08:00  14:59:55.000       6.83  6.84  6.85  6.71   
2026-01-09 14:59:58+08:00  14:59:58.000       6.83  6.84  6.85  6.71   
2026-01-09 15:00:01+08:00  15:00:01.000       6.86  6.84  6.86  6.71   

                           lastClose      amount  volume  pvolu

In [ ]:
jihe_df = df.between_time('9:15','9:26')
check_day = '2026-1-9'
tick_day = jihe_df.loc[check_day]
# open_price = 
#几个指标：开盘涨跌幅，集合竞价成交金额，集合竞价最后5s成交，最后
print(tick_day)

                                    time  lastPrice  open  high   low  \
time                                                                    
2026-01-09 09:15:01+08:00  1767921301000       0.00  0.00  0.00  0.00   
2026-01-09 09:15:04+08:00  1767921304000       0.00  0.00  0.00  0.00   
2026-01-09 09:15:07+08:00  1767921307000       0.00  0.00  0.00  0.00   
2026-01-09 09:15:10+08:00  1767921310000       0.00  0.00  0.00  0.00   
2026-01-09 09:15:13+08:00  1767921313000       0.00  0.00  0.00  0.00   
...                                  ...        ...   ...   ...   ...   
2026-01-09 09:24:43+08:00  1767921883000       0.00  0.00  0.00  0.00   
2026-01-09 09:24:46+08:00  1767921886000       0.00  0.00  0.00  0.00   
2026-01-09 09:24:55+08:00  1767921895000       0.00  0.00  0.00  0.00   
2026-01-09 09:24:58+08:00  1767921898000       0.00  0.00  0.00  0.00   
2026-01-09 09:25:01+08:00  1767921901000       6.84  6.84  6.84  6.84   

                           lastClose    amount  vo

## 4. 读取多个股票

In [8]:
# 读取多个股票（返回字典）
result = get_tick_data(["000001.SZ", "000002.SZ", "000004.SZ"], start_date="20260105")
for code, df in result.items():
    print(f"{code}: {len(df)} 条记录")

000001.SZ: 4811 条记录
000002.SZ: 4818 条记录
000004.SZ: 1832 条记录


In [1]:
for k,v in result.items():
    # jihe_df = v[v]
    v.index = pd.to_datetime(v['time'],unit='ms',utc=True).dt.tz_convert('Asia/Shanghai')
    jihe_df = v.between_time('9:15','9:26')
    print(k,jihe_df)
    break

NameError: name 'result' is not defined

In [ ]:
# 使用字符串指定多个股票
result = get_tick_data("000001.SZ,000002.SZ,000004.SZ", start_date="20260105")
for code, df in result.items():
    print(f"{code}: {len(df)} 条记录")

## 5. 数据字段说明

In [ ]:
# 查看数据字段
df = get_tick_data("000001.SZ", start_date="20260105")
print("Tick数据字段说明:")
field_desc = {
    'time': '时间戳(纳秒)',
    'datetime': '解析后的日期时间',
    'lastPrice': '最新价',
    'open': '开盘价',
    'high': '最高价',
    'low': '最低价',
    'lastClose': '昨收价',
    'volume': '成交量',
    'amount': '成交额',
    'pvolume': '流通成交量',
    'tickvol': 'Tick成交量',
    'bidPrice': '买盘价(5档)',
    'askPrice': '卖盘价(5档)',
    'bidVol': '买盘量(5档)',
    'askVol': '卖盘量(5档)',
}
for field, desc in field_desc.items():
    print(f"  {field}: {desc}")

## 6. 数据示例

In [ ]:
# 查看tick数据示例
df = get_tick_data("000001.SZ", start_date="20260105")
print("数据前10行:")
df[['datetime', 'lastPrice', 'volume', 'amount', 'bidPrice1', 'askPrice1']].head(10)

In [ ]:
# 查看五档行情
df = get_tick_data("000001.SZ", start_date="20260105")
print("五档买卖盘 (最后一笔):")
last_row = df.iloc[-1]
for i in range(1, 6):
    print(f"  买{i}: {last_row.get(f'bidPrice{i}', 'N/A'):.2f} x {last_row.get(f'bidVol{i}', 'N/A')}")
for i in range(1, 6):
    print(f"  卖{i}: {last_row.get(f'askPrice{i}', 'N/A'):.2f} x {last_row.get(f'askVol{i}', 'N/A')}")